# XMV-Net · Candidate E — Mixture-of-Experts Router

**Project:** XMV-Net · **Owner:** Sameen · **Report:** §2 Candidate E, §9 routing ablation

One small neural network per behavioural view, and a router that learns which expert should score each
customer. Outputs exactly what every other model in §7 outputs — per-fold AUC, AP, P@10%, R@10%, and an
out-of-fold prediction vector — plus the routing matrix the interpretability analyses need.

### The architecture

```
x  (127 standardized features)
│
├── recency columns ──►  gate ⊙ x ──► MLP ──► f_recency(x)     a logit, from recency features only
├── frequency columns ─► gate ⊙ x ──► MLP ──► f_frequency(x)
├── ... one expert per view (8 in total)
│
└── router: Linear(127 → 8) ──► top-k ──► g(x)                 weights, zero for unselected experts

logit(x) = Σ_{v ∈ top-k}  g_v(x) · f_v(x⁽ᵛ⁾)
```

**Two design choices carry the interpretability claim.**

1. **Input restriction.** Expert *v* only ever receives view *v*'s columns. It is architecturally incapable
   of seeing balance features when it is the recency expert, so "the recency expert scored this customer"
   describes the computation rather than estimating it.
2. **Mixture of predictions, not representations.** Each expert emits its own logit and the router weights
   them, so the prediction decomposes **exactly**: a customer's logit of 2.71 is recency contributing 1.92
   plus balance contributing 0.79, and nothing else. Fusing embeddings before a shared head (what XMV-Net
   does) is more expressive but gives no such decomposition.

The trade is real: no cross-view interaction before the decision. Given four model families already agree
within 0.0012 AUC on this data, that is expressiveness you almost certainly cannot use, traded for a
property you can defend.

### Coordination

This is XMV-Net's sibling: same per-view encoders, hard routing instead of gated attention. Positioned as
the **routing variant** it is ablation material for §9 and turns §2 Candidate E from "future work" into an
implemented result. Positioned as a separate model it duplicates Rafid's work — agree which before running.

### Session length

Roughly 2–3 minutes per fold on a T4, so **10–15 minutes** for the five folds.

## 1 · Locate the repo code

Resolves `data_loader.py` from the working directory, a cloned repository or an attached dataset. Warns when several copies are attached or the kernel holds a stale import.

In [ ]:
import sys
from pathlib import Path

CODE_DIR_OVERRIDE = None   # e.g. "/kaggle/input/datasets/jwuptr/xmvnet-code2" if several are attached

attached = ([p.parent for p in sorted(Path("/kaggle/input").glob("**/data_loader.py"))]
            if Path("/kaggle/input").exists() else [])
CANDIDATES = ([Path(CODE_DIR_OVERRIDE)] if CODE_DIR_OVERRIDE
              else [Path.cwd(), Path("/kaggle/working/Interpretable-Churn-Predictor"), *attached])

CODE_DIR = next((p for p in CANDIDATES if (p / "data_loader.py").exists()), None)
if CODE_DIR is None:
    raise FileNotFoundError(
        "data_loader.py not found. Attach the repo as a dataset, set CODE_DIR_OVERRIDE, or run:\n"
        "  !git clone <repo-url> /kaggle/working/Interpretable-Churn-Predictor"
    )
sys.path.insert(0, str(CODE_DIR))

import numpy as np
import pandas as pd
import sklearn
import data_loader as dl

if len(attached) > 1:
    print("WARNING: several code copies are attached -- set CODE_DIR_OVERRIDE if this picks the wrong one")
    for p in attached:
        print("   ", p)
if Path(dl.__file__).resolve().parent != Path(CODE_DIR).resolve():
    print(f"WARNING: data_loader was already imported from {dl.__file__}"
          " -- restart the kernel before trusting this run")

print("code:", CODE_DIR)
print("data_loader:", dl.__file__)
print("numpy", np.__version__, "| pandas", pd.__version__, "| scikit-learn", sklearn.__version__)

## 2 · Configuration

`TOP_K = 2` is the routing sparsity: each customer is scored by its two best experts. `TOP_K = 1` is hard
single-expert routing (the most extreme interpretability claim, the least stable to train); `TOP_K = 8`
makes it a dense mixture and the routing story disappears.

In [ ]:
SEED = 42
RUN_TAG = "moe"
MODEL_NAME = "moe"

TOP_K = 2                              # experts that score each customer
DROP_VIEWS = []                        # ["recency"] -> ablation A7
SAMPLE_ROWS = None                     # None = all 595,000 rows

HIDDEN, EMB, DROPOUT = 64, 32, 0.1     # per-expert MLP, matching XMV-Net's encoder shape
GATE_TAU = 1.0                         # temperature of the per-feature gate
NOISE_STD = 0.3                        # exploration noise on router logits (training only)

EPOCHS, BATCH, LR, WEIGHT_DECAY = 15, 1024, 1e-3, 1e-5
PATIENCE = 3                           # epochs without improvement before stopping
EARLY_STOP_FRACTION = 0.10             # inner slice of training rows, for early stopping

LAMBDA_AUX = 0.3                       # per-expert BCE: keeps unselected experts learning
LAMBDA_BALANCE = 0.05                  # load balancing: stops the router collapsing onto one expert
LAMBDA_GATE = 0.0                      # L1 on feature gates; raise for within-view sparsity

SAVE_CHECKPOINTS = True
TOP_FRACTION = 0.10

OUT_DIR = (Path("/kaggle/working/outputs/baselines") if Path("/kaggle/working").exists()
           else CODE_DIR / "outputs" / "baselines")
(OUT_DIR / "models").mkdir(parents=True, exist_ok=True)
print("outputs ->", OUT_DIR)

## 3 · Torch and device

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(SEED)
np.random.seed(SEED)
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch {torch.__version__} | device {DEV}")
if DEV.type == "cuda":
    print(" ", torch.cuda.get_device_name(0))

## 4 · Data and folds

Loads the cached feature matrix and the frozen 5-fold split. Verifies row order against the split's recorded account IDs and the column list against `frozen_folds_5.checks.json`.

In [ ]:
import json

train = dl.load_train(n_rows=SAMPLE_ROWS)


def find_file(name):
    """Works for the repo layout, a flat dataset upload, or any attached dataset."""
    return next((Path(p) for p in [
        CODE_DIR / "folds" / name,
        CODE_DIR / name,
        *(sorted(Path("/kaggle/input").glob(f"**/{name}")) if Path("/kaggle/input").exists() else []),
    ] if Path(p).exists()), None)


FOLDS_FILE = find_file("frozen_folds_5.npy")
if FOLDS_FILE is None:
    raise FileNotFoundError("frozen_folds_5.npy not found -- attach the repo or the folds dataset")

folds = dl.make_folds(train.y, ids=train.ids, frozen_path=FOLDS_FILE)
y = train.y.astype(np.int32)
N_SPLITS = int(folds.max()) + 1

CHECKS_FILE = find_file("frozen_folds_5.checks.json")
if CHECKS_FILE is None:
    print("FEATURE_COLS check skipped: frozen_folds_5.checks.json is not attached")
else:
    expected = json.loads(CHECKS_FILE.read_text())["matrix"]["feature_cols"]
    assert train.feature_cols == expected, "FEATURE_COLS differ from the verified list"
    print("FEATURE_COLS match the verified list:", CHECKS_FILE)

print(f"{len(y):,} rows | {train.n_features} columns | {N_SPLITS} folds | churn {y.mean():.5f}")
print(train.summary().to_string(index=False))

## 5 · Metrics

AUC, average precision, and precision/recall within the top 10% of predicted risk. Recall@10% is bounded at 78.9%: 11,900 customers selected against 15,087 churners per fold.

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

METRIC_NAMES = ["auc", "ap", "p_at_10", "r_at_10"]


def evaluate(y_true, scores):
    k = max(1, int(round(TOP_FRACTION * len(y_true))))
    top_k = np.argpartition(-scores, k - 1)[:k]
    hits = int(y_true[top_k].sum())
    return {
        "auc": float(roc_auc_score(y_true, scores)),
        "ap": float(average_precision_score(y_true, scores)),
        "p_at_10": hits / k,
        "r_at_10": hits / max(1, int(y_true.sum())),
    }

## 6 · Columns and scaling

Applies `DROP_VIEWS` through the loader's view groups (`["recency"]` gives ablation A7). Standardisation is fitted on training rows only.

In [ ]:
def select_columns(train, drop_views):
    keep = np.ones(train.n_features, dtype=bool)
    for view in drop_views:
        if view not in train.view_groups:
            raise KeyError(f"unknown view {view!r}; available: {list(train.view_groups)}")
        keep[train.view_groups[view]] = False
    idx = np.where(keep)[0]
    return idx, [train.feature_cols[i] for i in idx]


def standardize(Xtr, *others):
    mean = Xtr.mean(axis=0, dtype=np.float64)
    std = Xtr.std(axis=0, dtype=np.float64)
    std[std < 1e-6] = 1.0
    return [((X - mean) / std).astype(np.float32) for X in (Xtr, *others)]


COL_IDX, COLS = select_columns(train, DROP_VIEWS)
print(f"{len(COLS)} feature columns"
      + (f"  (dropped views: {', '.join(DROP_VIEWS)})" if DROP_VIEWS else ""))

## 7 · View slices

Maps each view to its column positions **within the selected columns**, so `DROP_VIEWS` shifts everything
correctly. A view that loses all its columns disappears from the expert list rather than becoming an
expert with no input.

In [ ]:
position = {c: i for i, c in enumerate(COLS)}
VIEW_SLICES = {}
for view, idx in train.view_groups.items():
    kept = [position[train.feature_cols[i]] for i in idx
            if train.feature_cols[i] in position]
    if kept:
        VIEW_SLICES[view] = np.asarray(kept, dtype=np.int64)

VIEW_NAMES = list(VIEW_SLICES)
print(f"{len(VIEW_NAMES)} experts, top-{TOP_K} routing")
for v, idx in VIEW_SLICES.items():
    print(f"  {v:<12} {len(idx):>3} features")

## 8 · The model

`ViewExpert` is one behavioural specialist: a learned feature gate, a two-layer MLP, and its own linear
head producing a single logit from its view's columns alone.

`MoERouter` scores all experts, then routes. Every expert is *computed* (they are tiny, and the auxiliary
loss needs their logits), but only the top-k receive non-zero weight, so the contribution of an unselected
expert to the prediction is exactly zero.

The router is deliberately a **single linear layer**: its weight matrix is 8 × 127 numbers you can read
directly to see which features push a customer toward which expert.

In [ ]:
class ViewExpert(nn.Module):
    """Feature gate -> 2-layer MLP -> one logit, from a single view's columns."""

    def __init__(self, in_dim, hidden=HIDDEN, emb=EMB, dropout=DROPOUT, tau=GATE_TAU):
        super().__init__()
        self.w = nn.Parameter(torch.ones(in_dim))
        self.tau = tau
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.BatchNorm1d(hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, emb), nn.BatchNorm1d(emb), nn.GELU(), nn.Dropout(dropout),
        )
        self.head = nn.Linear(emb, 1)

    def mask(self):
        return torch.sigmoid(self.w / self.tau)

    def forward(self, x):
        return self.head(self.net(x * self.mask())).squeeze(-1)


class MoERouter(nn.Module):
    """Per-view experts + a linear top-k router. logit = sum_{v in top-k} g_v * f_v."""

    def __init__(self, view_slices, in_dim, k=TOP_K, noise_std=NOISE_STD):
        super().__init__()
        self.view_names = list(view_slices)
        self.k = min(k, len(self.view_names))
        self.noise_std = noise_std
        for name, idx in view_slices.items():
            self.register_buffer(f"idx_{name}", torch.as_tensor(idx, dtype=torch.long))
        self.experts = nn.ModuleDict({n: ViewExpert(len(i)) for n, i in view_slices.items()})
        self.router = nn.Linear(in_dim, len(self.view_names))

    def forward(self, x):
        expert_logits = torch.stack(
            [self.experts[n](x[:, getattr(self, f"idx_{n}")]) for n in self.view_names], dim=1)

        route = self.router(x)
        if self.training and self.noise_std > 0:
            route = route + self.noise_std * torch.randn_like(route)

        top_val, top_idx = route.topk(self.k, dim=1)
        weights = torch.zeros_like(route).scatter(1, top_idx, F.softmax(top_val, dim=1))

        return {"logit": (weights * expert_logits).sum(dim=1),
                "weights": weights, "expert_logits": expert_logits}


def moe_loss(out, y, model):
    """BCE on the routed logit + per-expert BCE + load balancing (+ optional gate L1)."""
    bce = F.binary_cross_entropy_with_logits(out["logit"], y)
    aux = F.binary_cross_entropy_with_logits(
        out["expert_logits"], y.unsqueeze(1).expand_as(out["expert_logits"]))

    importance = out["weights"].sum(0)                       # how much work each expert got
    balance = (importance.std(unbiased=False) / (importance.mean() + 1e-8)) ** 2

    gate = torch.zeros((), device=bce.device)
    if LAMBDA_GATE:
        gate = torch.stack([e.mask().mean() for e in model.experts.values()]).mean()

    total = bce + LAMBDA_AUX * aux + LAMBDA_BALANCE * balance + LAMBDA_GATE * gate
    return total, {"bce": bce.item(), "aux": aux.item(),
                   "balance": balance.item(), "gate": float(gate)}


probe = MoERouter(VIEW_SLICES, len(COLS)).to(DEV)
print(f"parameters: {sum(p.numel() for p in probe.parameters()):,} "
      f"(router {probe.router.weight.numel() + probe.router.bias.numel():,})")
del probe

## 9 · Training one fold

Manual batching with the fold's tensors resident on the GPU — at 218 MB it fits comfortably and avoids
DataLoader overhead. Early stopping watches AUC on an inner 10% slice of the **training** rows, never the
scoring fold, exactly as the tree baselines do.

In [ ]:
import time
from sklearn.model_selection import train_test_split


@torch.no_grad()
def predict(model, X, batch=8192):
    model.eval()
    scores, weights = [], []
    for s in range(0, len(X), batch):
        out = model(X[s:s + batch])
        scores.append(torch.sigmoid(out["logit"]).cpu())
        weights.append(out["weights"].cpu())
    return torch.cat(scores).numpy(), torch.cat(weights).numpy()


def train_fold(Xtr_np, ytr_np, Xva_np):
    inner_fit, inner_es = train_test_split(np.arange(len(ytr_np)), test_size=EARLY_STOP_FRACTION,
                                           stratify=ytr_np, random_state=SEED)
    Xfit = torch.from_numpy(Xtr_np[inner_fit]).to(DEV)
    yfit = torch.from_numpy(ytr_np[inner_fit].astype(np.float32)).to(DEV)
    Xes = torch.from_numpy(Xtr_np[inner_es]).to(DEV)
    yes_np = ytr_np[inner_es]
    Xva = torch.from_numpy(Xva_np).to(DEV)

    model = MoERouter(VIEW_SLICES, Xfit.shape[1]).to(DEV)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    best_auc, best_epoch, best_state, stale = -1.0, 0, None, 0
    for epoch in range(1, EPOCHS + 1):
        model.train()
        order = torch.randperm(len(Xfit), device=DEV)
        parts = None
        for s in range(0, len(order), BATCH):
            batch = order[s:s + BATCH]
            if len(batch) < 2:            # BatchNorm needs more than one row
                continue
            loss, parts = moe_loss(model(Xfit[batch]), yfit[batch], model)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()

        es_scores, es_weights = predict(model, Xes)
        auc = roc_auc_score(yes_np, es_scores)
        effective = np.exp(-(es_weights * np.log(es_weights + 1e-12)).sum(1)).mean()
        print(f"    epoch {epoch:>2}  es-auc {auc:.5f}  bce {parts['bce']:.4f}  "
              f"balance {parts['balance']:.4f}  effective-views {effective:.2f}")

        if auc > best_auc + 1e-5:
            best_auc, best_epoch, stale = auc, epoch, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            stale += 1
            if stale >= PATIENCE:
                print(f"    early stop at epoch {epoch} (best {best_epoch})")
                break

    model.load_state_dict(best_state)
    va_scores, va_weights = predict(model, Xva)
    del Xfit, yfit, Xes, Xva
    if DEV.type == "cuda":
        torch.cuda.empty_cache()
    return model, va_scores, va_weights, best_epoch

## 10 · The fold loop

Same frozen folds, same metrics, same output layout as every other model — this row drops straight into
the §7 table. It additionally stores the out-of-fold **routing matrix**: one row per customer, one column
per expert, which is the input to every analysis in `report/section2_candidate_e.md`.

In [ ]:
rows = []
oof = np.full(len(y), np.nan)
oof_routing = np.full((len(y), len(VIEW_NAMES)), np.nan, dtype=np.float32)

for fold in range(N_SPLITS):
    tr_idx = np.where(folds != fold)[0]
    va_idx = np.where(folds == fold)[0]
    Xtr = np.asarray(train.X[tr_idx])[:, COL_IDX]
    Xva = np.asarray(train.X[va_idx])[:, COL_IDX]
    Xtr, Xva = standardize(Xtr, Xva)          # train rows only
    ytr, yva = y[tr_idx], y[va_idx]

    print(f"fold {fold}")
    t0 = time.perf_counter()
    model, scores, weights, best_epoch = train_fold(Xtr, ytr, Xva)
    seconds = time.perf_counter() - t0

    oof[va_idx] = scores
    oof_routing[va_idx] = weights
    metrics = evaluate(yva, scores)

    checkpoint = None
    if SAVE_CHECKPOINTS:
        checkpoint = f"{MODEL_NAME}_fold{fold}.pt"
        torch.save({"state_dict": model.state_dict(),
                    "view_slices": {k: v.tolist() for k, v in VIEW_SLICES.items()},
                    "feature_cols": COLS, "top_k": TOP_K,
                    "arch": {"hidden": HIDDEN, "emb": EMB, "dropout": DROPOUT, "tau": GATE_TAU}},
                   OUT_DIR / "models" / checkpoint)

    rows.append({"model": MODEL_NAME, "fold": fold, **metrics, "seconds": round(seconds, 1),
                 "best_iteration": best_epoch, "n_features": len(COLS), "checkpoint": checkpoint})
    print(f"  -> auc {metrics['auc']:.5f}  ap {metrics['ap']:.5f}  "
          f"p@10 {metrics['p_at_10']:.4f}  r@10 {metrics['r_at_10']:.4f}  {seconds:.1f}s\n")
    del Xtr, Xva, model

results = pd.DataFrame(rows)
results.to_csv(OUT_DIR / f"results_per_fold_{RUN_TAG}.csv", index=False)
np.save(OUT_DIR / f"oof_{MODEL_NAME}.npy", oof)
np.save(OUT_DIR / f"oof_routing_{MODEL_NAME}.npy", oof_routing)
display(results)

agg = results[METRIC_NAMES].agg(["mean", "std"])
for m in METRIC_NAMES:
    print(f"{m:<9} {agg.loc['mean', m]:.5f} +/- {agg.loc['std', m]:.5f}")

## 11 · Routing interpretability

The payoff. Every number here is exact: the weights are what the model used, and the contributions sum to
the logit it predicted.

Watch the collapse check first — if one expert wins almost every customer, the routing story is dead
regardless of AUC, and `LAMBDA_BALANCE` needs raising.

In [ ]:
R = oof_routing
effective = np.exp(-(R * np.log(R + 1e-12)).sum(1))
dominant = R.argmax(1)

print(f"effective experts per customer: mean {effective.mean():.2f}  "
      f"median {np.median(effective):.2f}   (1 = one expert, {len(VIEW_NAMES)} = uniform)\n")

coverage = pd.Series([VIEW_NAMES[i] for i in dominant]).value_counts(normalize=True)
print("share of customers each expert wins:")
print(coverage.round(4).to_string())
if coverage.max() > 0.9:
    print("\n  EXPERT COLLAPSE: one expert wins almost everything -- raise LAMBDA_BALANCE")

print("\nchurn rate by dominant expert:")
print(pd.DataFrame({"expert": [VIEW_NAMES[i] for i in dominant], "churn": y})
        .groupby("expert")["churn"].agg(["size", "mean"]).round(4).to_string())

print("\nmean routing weight per expert:")
print(pd.Series(R.mean(0), index=VIEW_NAMES).round(4).to_string())

routing_summary = {
    "effective_experts_mean": float(effective.mean()),
    "effective_experts_median": float(np.median(effective)),
    "coverage": {k: float(v) for k, v in coverage.items()},
    "mean_weight": {v: float(w) for v, w in zip(VIEW_NAMES, R.mean(0))},
}

## 12 · Exact decomposition for individual customers

What no tree model can produce: the prediction, broken into per-expert contributions that add up to it
arithmetically. These are the §10 case studies.

In [ ]:
ckpt = torch.load(OUT_DIR / "models" / f"{MODEL_NAME}_fold0.pt", map_location=DEV, weights_only=False)
model = MoERouter(VIEW_SLICES, len(COLS), k=ckpt["top_k"]).to(DEV)
model.load_state_dict(ckpt["state_dict"])
model.eval()

va_idx = np.where(folds == 0)[0]
tr_idx = np.where(folds != 0)[0]
Xtr_fold0 = np.asarray(train.X[tr_idx])[:, COL_IDX]      # only to recover this fold's scaling
Xva = np.asarray(train.X[va_idx])[:, COL_IDX]
_, Xva = standardize(Xtr_fold0, Xva)
del Xtr_fold0

with torch.no_grad():
    out = model(torch.from_numpy(Xva[:4096]).to(DEV))
weights = out["weights"].cpu().numpy()
expert_logits = out["expert_logits"].cpu().numpy()
logits = out["logit"].cpu().numpy()

order = np.argsort(-logits)[:3]        # three highest-risk customers in the sample
for i in order:
    p = 1 / (1 + np.exp(-logits[i]))
    print(f"customer {train.ids[va_idx[i]]}   p(churn) {p:.4f}   logit {logits[i]:+.3f}   "
          f"actual {int(y[va_idx[i]])}")
    for v in np.argsort(-weights[i]):
        if weights[i, v] > 0:
            print(f"    {VIEW_NAMES[v]:<12} weight {weights[i, v]:.3f} x expert logit "
                  f"{expert_logits[i, v]:+.3f}  ->  {weights[i, v] * expert_logits[i, v]:+.3f}")
    print(f"    {'sum':<12} {'':22} {(weights[i] * expert_logits[i]).sum():+.3f}\n")

## 13 · Run record

In [ ]:
import json

folds_manifest_path = Path(FOLDS_FILE).with_suffix(".json")
folds_manifest = json.loads(folds_manifest_path.read_text()) if folds_manifest_path.exists() else {}

(OUT_DIR / f"run_{RUN_TAG}.json").write_text(json.dumps({
    "run_tag": RUN_TAG, "model": MODEL_NAME, "architecture": "mixture of predictions, top-k routing",
    "experts": VIEW_NAMES, "top_k": TOP_K, "drop_views": DROP_VIEWS, "sample_rows": SAMPLE_ROWS,
    "seed": SEED, "n_features": len(COLS),
    "hyperparameters": {"hidden": HIDDEN, "emb": EMB, "dropout": DROPOUT, "gate_tau": GATE_TAU,
                        "noise_std": NOISE_STD, "epochs": EPOCHS, "batch": BATCH, "lr": LR,
                        "weight_decay": WEIGHT_DECAY, "patience": PATIENCE,
                        "lambda_aux": LAMBDA_AUX, "lambda_balance": LAMBDA_BALANCE,
                        "lambda_gate": LAMBDA_GATE},
    "routing": routing_summary,
    "folds": {"file": Path(FOLDS_FILE).name, "seed": folds_manifest.get("seed"),
              "ids_sha256": folds_manifest.get("ids_sha256")},
    "versions": {"numpy": np.__version__, "pandas": pd.__version__,
                 "scikit-learn": sklearn.__version__, "torch": torch.__version__},
    "summary": {m: {"mean": float(agg.loc["mean", m]), "std": float(agg.loc["std", m])}
                for m in METRIC_NAMES},
}, indent=2))

print("saved ->", OUT_DIR)
print(f"  results_per_fold_{RUN_TAG}.csv, oof_{MODEL_NAME}.npy, "
      f"oof_routing_{MODEL_NAME}.npy, run_{RUN_TAG}.json")

## 14 · Package outputs for download

Bundles the checkpoints, result table, out-of-fold predictions and the routing matrix into one archive.
Save Version before downloading; files left in `/kaggle/working` are discarded when the session ends.

In [ ]:
import zipfile

zip_path = OUT_DIR.parent / f"{MODEL_NAME}_outputs.zip"
artifacts = [f"results_per_fold_{RUN_TAG}.csv", f"run_{RUN_TAG}.json",
             f"oof_{MODEL_NAME}.npy", f"oof_routing_{MODEL_NAME}.npy"]

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted((OUT_DIR / "models").glob(f"{MODEL_NAME}_fold*.pt")):
        z.write(p, f"models/{p.name}")
    for name in artifacts:
        p = OUT_DIR / name
        if p.exists():
            z.write(p, name)

print(f"{zip_path}  ({zip_path.stat().st_size / 1e6:.1f} MB)\n")
for info in sorted(zipfile.ZipFile(zip_path).infolist(), key=lambda i: i.filename):
    print(f"  {info.filename:<38} {info.file_size / 1e6:7.2f} MB")
print("\nSave Version, then download from the finished version's Output tab.")